In [1]:
%pip install -U ripser persim

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 826.5/826.5 KB 6.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 KB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 51.2 MB/s eta 0:00:00a 0:00:01
  Using cached hopcroftkarp-1.2.5-py2.py3-none-any.whl
Note: you may need to restart the kernel to use updated packages.


In [2]:
# ============================================================
# 0. Auto-Fix for OLMo Dependencies
# ============================================================
def check_olmo_dependency():
    """Automatically installs ai2-olmo if missing, required for OLMo checkpoints."""
    try:
        import hf_olmo
        print("✅ Found hf_olmo module (ai2-olmo package).")
    except ImportError:
        print("⚠️ 'hf_olmo' module not found. Installing 'ai2-olmo' package for compatibility...")
        try:
            # The package name is 'ai2-olmo', NOT 'hf_olmo'
            subprocess.check_call([sys.executable, "-m", "pip", "install", "ai2-olmo"])
            print("✅ Successfully installed ai2-olmo.")
        except Exception as e:
            print(f"❌ Failed to auto-install ai2-olmo: {e}")
            print("Please run manually: pip install ai2-olmo")
            sys.exit(1)

# Check dependencies immediately
check_olmo_dependency()

⚠️ 'hf_olmo' module not found. Installing 'ai2-olmo' package for compatibility...
❌ Failed to auto-install ai2-olmo: name 'subprocess' is not defined
Please run manually: pip install ai2-olmo


NameError: name 'sys' is not defined

In [ ]:
import os
import gc
import sys
import subprocess
import random
import re
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from dataclasses import dataclass
from typing import List, Dict, Optional
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from huggingface_hub import list_repo_refs
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# ============================================================
# 0. Auto-Fix for OLMo Dependencies
# ============================================================
def check_deps():
    try:
        import hf_olmo
    except ImportError:
        print("⚠️ 'ai2-olmo' package not found. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "ai2-olmo"])
        print("✅ Installed ai2-olmo.")

check_deps()

try:
    from ripser import ripser
    HAS_RIPSER = True
except ImportError:
    HAS_RIPSER = False

# ============================================================
# 1. Configuration
# ============================================================

@dataclass
class ModelCfg:
    model_id: str
    short_name: str
    revision: str = "main"
    dtype: torch.dtype = torch.bfloat16
    trust_remote_code: bool = True
    attn_implementation: str = "eager"

@dataclass
class ExpConfig:
    n_samples: int = 500       
    batch_size: int = 4
    max_length: int = 128
    seed: int = 42
    local_csv_impossible: str = "impossibleQ.csv"
    local_csv_factual: str = "factual.csv"
    layers_to_scan: Optional[List[int]] = None 
    compute_geometry: bool = True  
    compute_topology: bool = True  
    compute_mechanics: bool = True 
    lid_k: int = 30
    advanced_subset_size: int = 32

def get_available_revisions(repo_id: str) -> Dict[int, str]:
    print(f"🔍 Fetching revisions for {repo_id}...")
    try:
        refs = list_repo_refs(repo_id)
        step_map = {}
        for branch in refs.branches:
            name = branch.name
            match = re.search(r"step(\d+)", name)
            if match:
                step_num = int(match.group(1))
                step_map[step_num] = name
        return step_map
    except Exception as e:
        print(f"❌ Failed to list revisions: {e}")
        return {}

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ============================================================
# 2. Data Loading & Input Helper
# ============================================================

def prepare_inputs(tokenizer, texts, device, max_len=128):
    """
    Tokenizes and removes 'token_type_ids' to prevent OLMo crashes.
    """
    inputs = tokenizer(
        texts, 
        padding=True, 
        truncation=True, 
        max_length=max_len, 
        return_tensors="pt"
    )
    if "token_type_ids" in inputs:
        del inputs["token_type_ids"]
    return inputs.to(device)

def get_real_datasets(cfg: ExpConfig) -> Dict[str, List[str]]:
    set_seed(cfg.seed)
    datasets_out = {}
    print(f"Loading datasets (Target: {cfg.n_samples})...")

    def load_local_csv(path):
        if os.path.exists(path):
            try:
                df = pd.read_csv(path)
                col = next((c for c in df.columns if 'question' in c.lower() or 'text' in c.lower()), None)
                if col:
                    raw = df[col].astype(str).tolist()
                    if len(raw) > cfg.n_samples:
                        raw = list(np.random.choice(raw, cfg.n_samples, replace=False))
                    return [f"Question: {t}\nAnswer:" for t in raw]
            except: pass
        return []

    datasets_out['1_Factual'] = load_local_csv(cfg.local_csv_factual)
    datasets_out['2_Impossible'] = load_local_csv(cfg.local_csv_impossible)

    pool = []
    try:
        ds = load_dataset("truthful_qa", "generation", split="validation")
        pool.extend([r['question'] for r in ds])
    except: pass
    try:
        ds = load_dataset("akariasai/PopQA", split="test")
        pool.extend(ds.to_pandas().sort_values("s_pop")['question'].tolist()[:1000])
    except: pass
    
    if pool:
        random.shuffle(pool)
        datasets_out['3_Hallucination'] = [f"Question: {q}\nAnswer:" for q in pool[:cfg.n_samples]]
    else:
        datasets_out['3_Hallucination'] = []

    return datasets_out

# ============================================================
# 3. Geometric Metrics
# ============================================================

class MetricEngine:
    @staticmethod
    def get_last_token_reps(hidden_states, attention_mask):
        last_indices = (attention_mask.sum(dim=1) - 1).clamp(min=0)
        return hidden_states[torch.arange(hidden_states.shape[0]), last_indices, :]

    @staticmethod
    def compute_lid(X, k=20):
        if len(X) <= k: return np.nan
        try:
            nn = NearestNeighbors(n_neighbors=k+1, n_jobs=-1).fit(X)
            dists, _ = nn.kneighbors(X)
            mask = dists[:, -1] > 1e-9
            if not np.any(mask): return 0.0
            lids = -k / np.sum(np.log(dists[mask, 1:] / dists[mask, -1][:, None] + 1e-10), axis=1)
            return np.mean(lids)
        except: return np.nan

    @staticmethod
    def compute_isotropy(X):
        if len(X) < 5: return np.nan
        try:
            pca = PCA(n_components=2).fit(X)
            return pca.explained_variance_ratio_[1] / (pca.explained_variance_ratio_[0] + 1e-9)
        except: return np.nan
    
    @staticmethod
    def compute_entropy(X):
        X_c = X - X.mean(axis=0)
        try:
            _, S, _ = np.linalg.svd(X_c, full_matrices=False)
            S_norm = S / (np.sum(S) + 1e-10)
            return np.exp(-np.sum(S_norm * np.log(S_norm + 1e-10)))
        except: return np.nan

    @staticmethod
    def compute_boundary_vector(X_pos, X_neg):
        vec = X_neg.mean(axis=0) - X_pos.mean(axis=0)
        norm = np.linalg.norm(vec)
        return vec / (norm + 1e-9), norm

    @staticmethod
    def compute_boundary_projection_stats(model, boundary_vec):
        b = torch.tensor(boundary_vec, device=model.device, dtype=model.dtype)
        with torch.no_grad():
            if hasattr(model, "get_output_embeddings"):
                logits = model.get_output_embeddings()(b)
            elif hasattr(model, "embed_out"): 
                logits = model.embed_out(b)
            else: return 0.0, 0.0
            probs = F.softmax(logits, dim=0)
            return -(probs * torch.log(probs + 1e-10)).sum().item(), probs.max().item()

# ============================================================
# 4. FULL Mechanistic Metrics (Patched for OLMo)
# ============================================================

def get_model_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"): return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "blocks"): return model.transformer.blocks
    if hasattr(model, "model") and hasattr(model.model, "transformer"): return model.model.transformer.blocks
    return model.transformer.h 

# --- 1. Fisher Information ---
def run_fisher(model, tokenizer, prompts, layer_idx, boundary_vec, device, batch_size=4):
    b_tens = torch.tensor(boundary_vec, device=device, dtype=torch.float64)
    layer = get_model_layers(model)[layer_idx]
    total_kl = 0.0; count = 0
    
    for i in range(0, len(prompts), batch_size):
        inputs = prepare_inputs(tokenizer, prompts[i:i+batch_size], device)
        with torch.no_grad():
            clean_out = model(**inputs)
            log_p_clean = F.log_softmax(clean_out.logits[:, -1, :].double(), dim=-1)
            p_clean = F.softmax(clean_out.logits[:, -1, :].double(), dim=-1)

        def hook(m, a, o):
            v = o[0] if isinstance(o, tuple) else o
            v = v.double(); v[:, -1, :] += 0.05 * b_tens
            return (v.to(model.dtype),) + o[1:] if isinstance(o, tuple) else v.to(model.dtype)

        h = layer.register_forward_hook(hook)
        with torch.no_grad(): pert_out = model(**inputs)
        h.remove()
        
        log_p_pert = F.log_softmax(pert_out.logits[:, -1, :].double(), dim=-1)
        p_pert = F.softmax(pert_out.logits[:, -1, :].double(), dim=-1)
        kl = 0.5 * (F.kl_div(log_p_clean, p_pert, reduction='batchmean', log_target=False) +
                    F.kl_div(log_p_pert, p_clean, reduction='batchmean', log_target=False))
        total_kl += kl.item()
        count += 1
    return (total_kl / max(count, 1)) / (0.05**2)

# --- 2. Hessian Curvature ---
def run_hessian(model, tokenizer, prompts, layer_idx, boundary_vec, device, batch_size=4):
    b_tens = torch.tensor(boundary_vec, device=device, dtype=torch.float64)
    layer = get_model_layers(model)[layer_idx]
    EPS = 3.0
    total_curv = 0.0; count = 0
    
    for i in range(0, len(prompts), batch_size):
        inputs = prepare_inputs(tokenizer, prompts[i:i+batch_size], device)
        
        def get_loss(shift):
            def hook(m, a, o):
                v = o[0] if isinstance(o, tuple) else o
                v = v.double(); v[:, -1, :] += shift * b_tens
                return (v.to(model.dtype),) + o[1:] if isinstance(o, tuple) else v.to(model.dtype)
            
            h = layer.register_forward_hook(hook)
            with torch.no_grad():
                logits = model(**inputs).logits[:, -1, :].double()
                loss = -F.log_softmax(logits, dim=-1).max(dim=-1).values
            h.remove()
            return loss
        
        L_c = get_loss(0.0)
        L_p = get_loss(EPS)
        L_m = get_loss(-EPS)
        curv = (L_p - 2 * L_c + L_m) / (EPS ** 2)
        total_curv += curv.mean().item()
        count += 1
    return total_curv / max(count, 1)

# --- 3. Jacobian Amplification ---
def run_jacobian(model, tokenizer, prompts, layer_idx, boundary_vec, device, epsilon=0.5, batch_size=4):
    b_tens = torch.tensor(boundary_vec, device=device, dtype=model.dtype)
    layer = get_model_layers(model)[layer_idx]
    total_amp = 0; count = 0
    
    for i in range(0, len(prompts), batch_size):
        inputs = prepare_inputs(tokenizer, prompts[i:i+batch_size], device)
        
        clean_out = []
        def clean_h(m, a, o): clean_out.append(o[0] if isinstance(o, tuple) else o)
        h1 = layer.register_forward_hook(clean_h)
        with torch.no_grad(): model(**inputs)
        h1.remove()
        
        pert_out = []
        def pre_h(m, args):
            h = args[0].clone()
            h[:, -1, :] += (epsilon * b_tens)
            return (h, *args[1:])
        def post_h(m, a, o): pert_out.append(o[0] if isinstance(o, tuple) else o)
        
        h2 = layer.register_forward_pre_hook(pre_h)
        h3 = layer.register_forward_hook(post_h)
        with torch.no_grad(): model(**inputs)
        h2.remove(); h3.remove()
        
        if clean_out and pert_out:
            diff = (pert_out[0][:, -1, :] - clean_out[0][:, -1, :]) / epsilon
            total_amp += torch.norm(diff, dim=-1).mean().item()
            count += 1
    return total_amp / max(count, 1)

# --- 4. Gradient Blockage ---
def run_grad_blockage(model, tokenizer, prompts, layer_idx, boundary_vec, device, batch_size=4):
    b_tens = torch.tensor(boundary_vec, device=device, dtype=torch.float32)
    target_words = [" I", " not", " unsure", " maybe", " depends", " unknown"]
    target_ids = []
    for w in target_words:
        ids = tokenizer(w, add_special_tokens=False).input_ids
        if ids: target_ids.append(ids[-1])
    
    if not target_ids: return 0.0
    
    layer = get_model_layers(model)[layer_idx]
    grads = []
    def hook(m, gi, go): grads.append(go[0][:, -1, :].detach().float())
    h = layer.register_full_backward_hook(hook)
    
    total = 0; count = 0
    for i in range(0, len(prompts), batch_size):
        inputs = prepare_inputs(tokenizer, prompts[i:i+batch_size], device)
        model.zero_grad()
        out = model(**inputs)
        loss = out.logits[:, -1, target_ids].sum()
        loss.backward()
        
        if grads:
            sims = F.cosine_similarity(grads[-1], b_tens.unsqueeze(0), dim=1)
            total += sims.mean().item()
            count += 1
            grads.clear()
    h.remove()
    return total / max(count, 1)

# --- 5. Loss Gradient Alignment ---
def run_loss_grad_align(model, tokenizer, prompts, layer_idx, boundary_vec, device, batch_size=4):
    b_tens = torch.tensor(boundary_vec, device=device, dtype=torch.float32)
    layer = get_model_layers(model)[layer_idx]
    grads = []
    
    def hook(m, gi, go): grads.append(go[0][:, -1, :].detach().float())
    h = layer.register_full_backward_hook(hook)
    
    total = 0; count = 0
    for i in range(0, len(prompts), batch_size):
        inputs = prepare_inputs(tokenizer, prompts[i:i+batch_size], device)
        model.zero_grad()
        model(**inputs, labels=inputs.input_ids).loss.backward()
        if grads:
            total += F.cosine_similarity(grads[-1], b_tens.unsqueeze(0)).mean().item()
            count += 1
            grads.clear()
    h.remove()
    return total / max(count, 1)

# --- 6. Vocab Alignment ---
def precompute_vocab_subspace(model, k=100):
    with torch.no_grad():
        if hasattr(model, "get_output_embeddings"):
            W_u = model.get_output_embeddings().weight.data.float()
        elif hasattr(model, "embed_out"): 
            W_u = model.embed_out.weight.data.float()
        else: return None
        try:
            _, _, Vh = torch.linalg.svd(W_u, full_matrices=False)
            return Vh[:k, :]
        except: return None

def run_vocab_align(boundary_vec, vocab_V):
    if vocab_V is None: return np.nan
    b = torch.tensor(boundary_vec, device=vocab_V.device, dtype=vocab_V.dtype)
    return (torch.mv(vocab_V, b).norm() / (b.norm() + 1e-9)).item()

# ============================================================
# 5. Main Loop
# ============================================================

def analyze_step(model_cfg, exp_cfg, datasets, label):
    print(f"\n🔬 Step: {label} (Rev: {model_cfg.revision})")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_cfg.model_id, trust_remote_code=True)
        if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
        
        model = AutoModelForCausalLM.from_pretrained(
            model_cfg.model_id, revision=model_cfg.revision, 
            torch_dtype=model_cfg.dtype, trust_remote_code=True, device_map="auto"
        )
        model.eval()
    except Exception as e:
        print(f"❌ Failed to load model: {e}"); return None, None

    # Precompute Vocab Subspace for this checkpoint
    vocab_V = None
    if exp_cfg.compute_mechanics:
        vocab_V = precompute_vocab_subspace(model)
        if vocab_V is not None: vocab_V = vocab_V.to(model.dtype)

    # Extraction
    layer_storage = {l: {k: [] for k in datasets} for l in exp_cfg.layers_to_scan}
    for bucket, texts in datasets.items():
        if not texts: continue
        for i in tqdm(range(0, len(texts), exp_cfg.batch_size), desc=f"  Extract {bucket}", leave=False):
            inp = prepare_inputs(tokenizer, texts[i:i+exp_cfg.batch_size], model.device, exp_cfg.max_length)
            with torch.no_grad():
                out = model(**inp, output_hidden_states=True)
                for l in exp_cfg.layers_to_scan:
                    hs = out.hidden_states[l+1]
                    rep = MetricEngine.get_last_token_reps(hs, inp.attention_mask)
                    layer_storage[l][bucket].append(rep.float().cpu().numpy())

    # Analysis
    results_geo, results_mech = [], []
    print(f"  Analysing {len(exp_cfg.layers_to_scan)} layers with full metrics...")
    
    for l in exp_cfg.layers_to_scan:
        data = {k: np.concatenate(v, axis=0) if v else np.array([]) for k,v in layer_storage[l].items()}
        
        # Geometry
        for b, X in data.items():
            if len(X) < 10: continue
            results_geo.append({
                "step": label, "layer": l, "bucket": b,
                "lid": MetricEngine.compute_lid(X, exp_cfg.lid_k),
                "isotropy": MetricEngine.compute_isotropy(X),
                "entropy": MetricEngine.compute_entropy(X)
            })

        # Mechanics (The Full Suite)
        pairs = [("1_Factual", "2_Impossible"), ("1_Factual", "3_Hallucination")]
        for pos, neg in pairs:
            if len(data.get(pos,[])) > 10 and len(data.get(neg,[])) > 10:
                vec, norm = MetricEngine.compute_boundary_vector(data[pos], data[neg])
                ent, mp = MetricEngine.compute_boundary_projection_stats(model, vec)
                
                # Evaluation subset
                sub_ds = datasets[neg][:exp_cfg.advanced_subset_size]
                
                row = {
                    "step": label, "layer": l, "boundary": neg, 
                    "norm": norm, "boundary_entropy": ent, "max_prob": mp
                }

                if exp_cfg.compute_mechanics:
                    # 1. Fisher
                    row["fisher"] = run_fisher(model, tokenizer, sub_ds, l, vec, model.device)
                    # 2. Hessian
                    row["hessian"] = run_hessian(model, tokenizer, sub_ds, l, vec, model.device)
                    # 3. Jacobian
                    row["jacobian"] = run_jacobian(model, tokenizer, sub_ds, l, vec, model.device)
                    # 4. Grad Blockage (Targeted)
                    row["grad_block"] = run_grad_blockage(model, tokenizer, sub_ds, l, vec, model.device)
                    # 5. Loss Grad Align (General)
                    row["grad_align"] = run_loss_grad_align(model, tokenizer, sub_ds, l, vec, model.device)
                    # 6. Vocab Align
                    row["vocab_align"] = run_vocab_align(vec, vocab_V)

                results_mech.append(row)

    del model, tokenizer; gc.collect(); torch.cuda.empty_cache()
    return pd.DataFrame(results_geo), pd.DataFrame(results_mech)

if __name__ == "__main__":
    cfg = ExpConfig(n_samples=200, batch_size=4, layers_to_scan=None) 
    model_id = "allenai/OLMo-7B"
    
    # 1. Get Datasets
    data = get_real_datasets(cfg)
    if not data['1_Factual']: exit("No data found.")

    # 2. Find Revision Names
    rev_map = get_available_revisions(model_id)
    print(f"✅ Found {len(rev_map)} revisions.")
    
    # 3. Match user steps
    target_steps = [2000, 20000, 50000]
    # 
    steps_to_run = []
    
    for t in target_steps:
        if t in rev_map:
            steps_to_run.append((t, rev_map[t]))
        else:
            available = sorted(rev_map.keys())
            if available:
                closest = min(available, key=lambda x: abs(x-t))
                steps_to_run.append((closest, rev_map[closest]))

    steps_to_run.append(("Final", "main"))
    steps_to_run = list(dict.fromkeys(steps_to_run))

    # 4. Run
    all_g, all_m = [], []
    if cfg.layers_to_scan is None:
        cfg.layers_to_scan = list(range(0, 32, 2)) # Default scan

    for label, rev in steps_to_run:
        m_cfg = ModelCfg(model_id, "OLMo-7B", revision=rev)
        df_g, df_m = analyze_step(m_cfg, cfg, data, str(label))
        if df_g is not None:
            all_g.append(df_g); all_m.append(df_m)
            df_g.to_csv(f"geo_temp_{label}.csv")
            df_m.to_csv(f"mech_temp_{label}.csv") # Save mechanics too!

    if all_g:
        pd.concat(all_g).to_csv("OLMo_Geometry_Final.csv", index=False)
        pd.concat(all_m).to_csv("OLMo_Mechanics_Final.csv", index=False)
        print("Done.")

In [ ]:
import os
import gc
import sys
import subprocess
import random
import re
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from dataclasses import dataclass
from typing import List, Dict, Optional
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from datasets import load_dataset
from huggingface_hub import list_repo_refs

# ============================================================
# 0. Dependencies & Setup
# ============================================================
def check_deps():
    try:
        import hf_olmo
    except ImportError:
        print("⚠️ 'ai2-olmo' package not found. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "ai2-olmo"])
        print("✅ Installed ai2-olmo.")

check_deps()

# ============================================================
# 1. Configuration
# ============================================================

@dataclass
class ModelCfg:
    model_id: str
    revision: str = "main"
    dtype: torch.dtype = torch.float16
    is_step_0: bool = False

@dataclass
class ExpConfig:
    n_samples: int = 100       
    batch_size: int = 8
    max_length: int = 64
    seed: int = 42
    local_csv_impossible: str = "impossibleQ.csv"
    local_csv_factual: str = "factual.csv"
    layers_to_scan: Optional[List[int]] = None 
    vocab_subspace_k: int = 200

# ============================================================
# 2. Data Loading
# ============================================================

def prepare_inputs(tokenizer, texts, device, max_len=128):
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
    if "token_type_ids" in inputs: del inputs["token_type_ids"]
    return inputs.to(device)

def get_real_datasets(cfg: ExpConfig) -> Dict[str, List[str]]:
    print(f"Loading datasets (Target: {cfg.n_samples})...")
    datasets_out = {}

    def load_local(path):
        if os.path.exists(path):
            try:
                df = pd.read_csv(path)
                col = next((c for c in df.columns if 'text' in c.lower() or 'question' in c.lower()), None)
                if col: return [f"Question: {x}\nAnswer:" for x in df[col].dropna().astype(str).tolist()]
            except: pass
        return []

    datasets_out['1_Factual'] = load_local(cfg.local_csv_factual)
    datasets_out['2_Impossible'] = load_local(cfg.local_csv_impossible)

    if not datasets_out['1_Factual']:
        print("  ⚠️ Local factual.csv not found, using TruthfulQA fallback.")
        try:
            ds = load_dataset("truthful_qa", "generation", split="validation")
            datasets_out['1_Factual'] = [f"Question: {x}\nAnswer:" for x in ds['question'][:cfg.n_samples]]
        except: datasets_out['1_Factual'] = []
    
    if not datasets_out['2_Impossible']:
        print("  ⚠️ Local impossibleQ.csv not found, generating dummy data.")
        datasets_out['2_Impossible'] = [f"Question: What is the capital of Mars region {i}?\nAnswer:" for i in range(cfg.n_samples)]

    # Trim
    for k in datasets_out:
        if len(datasets_out[k]) > cfg.n_samples:
            datasets_out[k] = random.sample(datasets_out[k], cfg.n_samples)
            
    return datasets_out

# ============================================================
# 3. Robust Layer Extraction (Fixed for OLMo & Pythia)
# ============================================================

def get_model_layers(model):
    """
    Robustly finds the ModuleList containing the transformer layers.
    Works for OLMo, Pythia (GPT-NeoX), Llama, etc.
    """
    # 1. Try Specific Known Paths
    paths = [
        ("model", "layers"),                # Llama, Mistral
        ("model", "transformer", "blocks"), # OLMo (Standard)
        ("backbone", "blocks"),             # MosaicML
        ("gpt_neox", "layers"),             # Pythia / GPT-NeoX
        ("transformer", "h"),               # GPT-2
        ("transformer", "blocks"),          # Bert-like
    ]
    
    for path in paths:
        curr = model
        valid = True
        for part in path:
            if hasattr(curr, part):
                curr = getattr(curr, part)
            else:
                valid = False
                break
        if valid and isinstance(curr, (torch.nn.ModuleList, list)):
            return curr
            
    # 2. Fallback: Find largest ModuleList
    largest_list = None
    max_len = 0
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.ModuleList):
            if len(module) > max_len:
                largest_list = module
                max_len = len(module)
    if largest_list is not None: return largest_list
    raise AttributeError(f"Could not locate layer list in {type(model).__name__}")

# ============================================================
# 4. Metric Engines
# ============================================================

class MetricEngine:
    @staticmethod
    def get_last_token_reps(hidden_states, attention_mask):
        last_indices = (attention_mask.sum(dim=1) - 1).clamp(min=0)
        return hidden_states[torch.arange(hidden_states.shape[0]), last_indices, :]

    @staticmethod
    def compute_boundary_vector(X_pos, X_neg):
        p = torch.tensor(X_pos, dtype=torch.float64).mean(dim=0)
        n = torch.tensor(X_neg, dtype=torch.float64).mean(dim=0)
        vec = n - p
        norm = torch.norm(vec).item()
        return (vec / (norm + 1e-9)).float(), norm

def precompute_vocab_subspace(model, k=200):
    print("  ...computing Unembedding Subspace (SVD)...")
    with torch.no_grad():
        W_u = None
        # OLMo
        if hasattr(model, "get_output_embeddings"): W_u = model.get_output_embeddings().weight.data.float()
        elif hasattr(model, "embed_out"): W_u = model.embed_out.weight.data.float()
        
        if W_u is None: return None
        try:
            _, _, Vh = torch.linalg.svd(W_u, full_matrices=False)
            return Vh[:k, :].to(model.device) 
        except Exception as e:
            print(f"SVD Failed: {e}"); return None

def run_null_space_test(boundary_vec, vocab_V, device):
    if vocab_V is None: return {}
    b_tens = boundary_vec.to(device).float()
    vocab_V = vocab_V.to(device).float()
    
    # 1. Alignment with Output Subspace
    coeffs = torch.mv(vocab_V, b_tens)
    proj_norm = torch.norm(coeffs).item()
    
    # 2. Random Control
    rand_v = torch.randn_like(b_tens)
    rand_v = rand_v / torch.norm(rand_v)
    rand_coeffs = torch.mv(vocab_V, rand_v)
    rand_proj_norm = torch.norm(rand_coeffs).item()
    
    return {
        "null_align": proj_norm,
        "null_rand_align": rand_proj_norm,
        "null_ratio": proj_norm / (rand_proj_norm + 1e-9)
    }

def run_fisher(model, tokenizer, prompts, layer_idx, boundary_vec, device):
    b_tens = boundary_vec.to(device).double()
    layer = get_model_layers(model)[layer_idx]
    
    inputs = prepare_inputs(tokenizer, prompts[:4], device)
    
    with torch.no_grad():
        clean_logits = model(**inputs).logits[:, -1, :].double()
        p_clean = F.softmax(clean_logits, dim=-1)

    def hook(m, a, o):
        v = o[0] if isinstance(o, tuple) else o
        v = v.double(); v[:, -1, :] += 0.05 * b_tens
        return (v.to(model.dtype),) + o[1:] if isinstance(o, tuple) else v.to(model.dtype)

    h = layer.register_forward_hook(hook)
    with torch.no_grad(): pert_logits = model(**inputs).logits[:, -1, :].double()
    h.remove()
    
    log_p_pert = F.log_softmax(pert_logits, dim=-1)
    kl = F.kl_div(log_p_pert, p_clean, reduction='batchmean', log_target=False)
    return (kl.item()) / (0.05**2)

# ============================================================
# 5. Step Discovery Logic (Crucial for Early Training)
# ============================================================

def get_closest_revisions(model_id, targets: List[int]):
    """
    Finds the revisions in the repo that best match the target steps.
    Supports 'step100', 'step-100', 'global_step100' etc.
    """
    try:
        refs = list_repo_refs(model_id)
        branches = [b.name for b in refs.branches]
    except Exception as e:
        print(f"⚠️ Could not list branches: {e}")
        branches = ["main"]

    step_map = {}
    
    # Regex to extract step number from branch name
    # Looks for 'step' followed optionally by symbols, then digits
    regex = re.compile(r"step\D*(\d+)", re.IGNORECASE)

    for b in branches:
        if b == "main": continue
        match = regex.search(b)
        if match:
            s = int(match.group(1))
            step_map[s] = b
    
    final_plan = []
    
    # Always include Step 0 (Virtual) if requested
    if 0 in targets:
        final_plan.append({"step": 0, "rev": "random", "virtual": True})
    
    available_steps = sorted(step_map.keys())
    print(f"🔍 Found {len(available_steps)} revisions in repo: {available_steps[:10]} ...")

    for t in targets:
        if t == 0: continue
        
        # 1. Exact Match
        if t in step_map:
            final_plan.append({"step": t, "rev": step_map[t], "virtual": False})
        else:
            # 2. Closest Match (if within reasonable range)
            if not available_steps: continue
            closest = min(available_steps, key=lambda x: abs(x-t))
            # Only snap to closest if it's somewhat related (e.g. requested 10, got 16 is ok. requested 10 got 1000 is bad)
            if abs(closest - t) < t * 0.5 or (t < 1000 and closest < 1000):
                print(f"  Note: Target {t} not found. Using closest: {closest}")
                final_plan.append({"step": closest, "rev": step_map[closest], "virtual": False})
            else:
                print(f"  ⚠️ Skipping Target {t}: Closest available is {closest} (too far)")

    # Deduplicate
    seen = set()
    unique_plan = []
    for p in final_plan:
        if p["step"] not in seen:
            unique_plan.append(p)
            seen.add(p["step"])
            
    return unique_plan

# ============================================================
# 6. Main Execution
# ============================================================

def analyze_step(model_cfg, exp_cfg, datasets, label):
    print(f"\n🔬 Processing Step: {label}")
    
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_cfg.model_id, trust_remote_code=True)
        if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
        
        if model_cfg.is_step_0:
            print("  🎲 Initializing Random Model (Virtual Step 0)...")
            config = AutoConfig.from_pretrained(model_cfg.model_id, trust_remote_code=True)
            model = AutoModelForCausalLM.from_config(config, trust_remote_code=True)
            model.to("cuda" if torch.cuda.is_available() else "cpu")
            model = model.to(model_cfg.dtype)
        else:
            print(f"  📥 Loading revision: {model_cfg.revision}")
            model = AutoModelForCausalLM.from_pretrained(
                model_cfg.model_id, 
                revision=model_cfg.revision, 
                torch_dtype=model_cfg.dtype, 
                trust_remote_code=True, 
                device_map="auto"
            )
        model.eval()
    except Exception as e:
        print(f"❌ Failed to load model: {e}"); return None

    vocab_V = precompute_vocab_subspace(model, k=exp_cfg.vocab_subspace_k)
    all_layers = get_model_layers(model)
    n_layers = len(all_layers)
    
    # Auto-detect middle layers for scanning if not specified
    if exp_cfg.layers_to_scan is None:
        scan_indices = list(range(0, n_layers, 2))
    else:
        scan_indices = [x for x in exp_cfg.layers_to_scan if x < n_layers]

    # Extraction
    layer_storage = {l: {k: [] for k in datasets} for l in scan_indices}
    for bucket, texts in datasets.items():
        if not texts: continue
        texts = texts[:exp_cfg.n_samples]
        for i in tqdm(range(0, len(texts), exp_cfg.batch_size), desc=f"  Extract {bucket}", leave=False):
            inp = prepare_inputs(tokenizer, texts[i:i+exp_cfg.batch_size], model.device, exp_cfg.max_length)
            with torch.no_grad():
                out = model(**inp, output_hidden_states=True)
                for l in scan_indices:
                    hs = out.hidden_states[l+1] 
                    rep = MetricEngine.get_last_token_reps(hs, inp.attention_mask)
                    layer_storage[l][bucket].append(rep.float().cpu().numpy())

    # Analysis
    results = []
    for l in scan_indices:
        data = {k: np.concatenate(v, axis=0) if v else np.array([]) for k,v in layer_storage[l].items()}
        pairs = [("1_Factual", "2_Impossible")]
        
        for pos, neg in pairs:
            if len(data.get(pos,[])) > 5 and len(data.get(neg,[])) > 5:
                vec, norm = MetricEngine.compute_boundary_vector(data[pos], data[neg])
                null_stats = run_null_space_test(vec, vocab_V, model.device)
                
                fisher = 0.0
                try:
                    sub_ds = datasets[neg][:8]
                    fisher = run_fisher(model, tokenizer, sub_ds, l, vec, model.device)
                except: pass

                row = {"step": label, "layer": l, "magnitude": norm, "fisher_info": fisher}
                row.update(null_stats)
                results.append(row)

    del model, tokenizer; gc.collect(); torch.cuda.empty_cache()
    return pd.DataFrame(results)

if __name__ == "__main__":
    # --- CONFIGURATION ---
    # NOTE: OLMo-7B usually only has step 1000+. 
    # Use "EleutherAI/pythia-6.9b" if you need steps 10, 50, 100
    
    # MODEL_ID = "allenai/OLMo-7B" 
    MODEL_ID = "EleutherAI/pythia-6.9b" # <-- Uncomment this for real Step 10 data
    
    TARGET_STEPS = [8, 32, 64, 128, 256, 512, 1000]
    
    cfg = ExpConfig(n_samples=50, batch_size=8)
    data = get_real_datasets(cfg)
    
    # 1. Build Plan
    print(f"🔍 Looking for targets: {TARGET_STEPS} in {MODEL_ID}")
    plan = get_closest_revisions(MODEL_ID, TARGET_STEPS)
    
    if not plan:
        print("❌ No matching steps found. Check the model ID or manual list.")
        exit()

    # 2. Run
    all_res = []
    for p in plan:
        m_cfg = ModelCfg(MODEL_ID, revision=p["rev"], is_step_0=p["virtual"])
        df = analyze_step(m_cfg, cfg, data, str(p["step"]))
        if df is not None:
            all_res.append(df)
            df.to_csv(f"null_space_step_{p['step']}.csv", index=False)

    if all_res:
        pd.concat(all_res).to_csv("Early_NullSpace_Analysis.csv", index=False)
        print("\n✅ Done.")